# 🐻 H.U.G.H. SOVEREIGN BIRTH — COLAB SPEED-FUCK EDITION
**MISSION OBJECTIVE: ARC-AGI-3 (>50%)**

This notebook will perform a high-performance Unsloth fine-tune on your signed heritage data.

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "DavidAU/LFM2.5-1.2B-Thinking-Claude-4.6-Opus-Heretic-Uncensored-DISTILL",
    max_seq_length = 2048,
    load_in_4bit = True,
    token = "os.environ["HF_TOKEN"]"
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
import json
from datasets import Dataset

# ── UPLOAD YOUR DATA ──
# Manually upload 'hugh_personality_training_v3_strict.jsonl' to the Colab sidebar if it doesn't exist

with open('hugh_personality_training_v3_strict.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]

def format_fn(ex):
    text = ""
    for m in ex['messages']:
        text += f"<|{m['role']}|>\n{m['content']}\n"
    return {'text': text}

dataset = Dataset.from_list(data).map(format_fn)

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 10,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()
model.push_to_hub_merged("oldmangrizzz/hugh-sovereign-heretic-v1", tokenizer, save_method = "lora", token = "os.environ["HF_TOKEN"]")
print("MISSION COMPLETE. H.U.G.H. IS BORN.")